# Lab 3 — Build the Retrieval Pipeline (RAG)
**AI Engineering for Managers · Day 1 · 13:45–14:45**

**Goal:** build the service-knowledge assistant end to end: ingest the Axion corpus → chunk → embed → retrieve → answer with citations. Then **break it on purpose** and fix it, so you recognise the failure modes when your teams hit them.

In Lab 1 the models guessed about F-07 because they had never seen our manuals. Retrieval fixes that — when it's done properly.

In [ ]:
# SETUP — run this first. It is the same in every lab.
import os, time, json

from openai import OpenAI, AzureOpenAI

ENDPOINT       = os.environ.get("FOUNDRY_ENDPOINT", "")
API_KEY        = os.environ.get("FOUNDRY_API_KEY", "")
MODEL_MINI     = os.environ.get("MODEL_MINI", "gpt-4o-mini")          # small, cheap workhorse
MODEL_FRONTIER = os.environ.get("MODEL_FRONTIER", "gpt-4o")           # large, expensive
MODEL_EMBED    = os.environ.get("MODEL_EMBED", "text-embedding-3-small")

assert ENDPOINT and API_KEY, (
    "Model endpoint not configured. On the lab VMs these are pre-set — "
    "if you see this, call the trainer."
)

# Works with both Azure OpenAI endpoints and OpenAI-compatible serverless endpoints
if any(h in ENDPOINT for h in ("openai.azure.com", "cognitiveservices", "services.ai.azure.com")):
    client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=API_KEY, api_version="2024-10-21")
else:
    client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

print("Client ready.")
print("Mini model     :", MODEL_MINI)
print("Frontier model :", MODEL_FRONTIER)

In [ ]:
# Helpers used all day. with_retry keeps the room moving if the endpoint rate-limits briefly.
def with_retry(fn, tries=3, wait=5):
    for a in range(tries):
        try:
            return fn()
        except Exception as e:
            if a == tries - 1: raise
            print(f"  (retrying in {wait*(a+1)}s: {e})")
            time.sleep(wait * (a + 1))

def ask(model, messages, temperature=0.2, **kw):
    """Call a chat model and return text + latency + token counts."""
    t0 = time.perf_counter()
    r = with_retry(lambda: client.chat.completions.create(model=model, messages=messages,
                                                          temperature=temperature, **kw))
    latency = time.perf_counter() - t0
    u = r.usage
    return {
        "model": model,
        "text": r.choices[0].message.content,
        "latency_s": round(latency, 2),
        "input_tokens": u.prompt_tokens,
        "output_tokens": u.completion_tokens,
    }

# Prices in USD per 1 MILLION tokens.
# >>> EDIT these to match the actual prices of your deployments. <<<
PRICE = {
    MODEL_MINI:     {"in": 0.15, "out": 0.60},
    MODEL_FRONTIER: {"in": 2.50, "out": 10.00},
}

def cost_usd(result):
    p = PRICE[result["model"]]
    return round((result["input_tokens"] * p["in"] +
                  result["output_tokens"] * p["out"]) / 1_000_000, 6)

## 3.1 — Ingest the corpus
14 documents: manuals, fault-code references, service bulletins, firmware release notes, commissioning notes. All synthetic — no real customer data anywhere in this room.

In [ ]:
from pathlib import Path

CORPUS_DIR = Path("../corpus") if Path("../corpus").exists() else Path("corpus")
docs = {p.name: p.read_text(encoding="utf-8") for p in sorted(CORPUS_DIR.glob("*.md"))}
print(f"{len(docs)} documents, {sum(len(t) for t in docs.values()):,} characters")
for name in docs: print(" -", name)

## 3.2 — Chunk
Models can't usefully attend to 15 whole manuals per question, and you pay for every token you send. So we split documents into chunks. **Chunk size is the first dial your teams will get wrong** — we start with a sensible default and later we'll see what a bad one does.

In [ ]:
def chunk_document(name, text, target_chars=1200, overlap=150):
    """Split on paragraph boundaries, pack into ~target_chars chunks with a little overlap."""
    paras, chunks, buf = [p for p in text.split("\n\n") if p.strip()], [], ""
    for p in paras:
        if len(buf) + len(p) > target_chars and buf:
            chunks.append(buf.strip()); buf = buf[-overlap:] + "\n\n"
        buf += p + "\n\n"
    if buf.strip(): chunks.append(buf.strip())
    return [{"doc": name, "chunk_id": f"{name}#{i}", "text": c} for i, c in enumerate(chunks)]

CHUNK_SIZE = 1200   # <- the dial
chunks = [c for name, text in docs.items() for c in chunk_document(name, text, CHUNK_SIZE)]
print(f"{len(chunks)} chunks (target {CHUNK_SIZE} chars)")
print("\n--- sample chunk ---\n", chunks[10]["text"][:400])

## 3.3 — Embed
Each chunk becomes a vector — a point in space where *similar meaning = nearby*. One embedding call per batch; this is the only "indexing cost" of the whole system, and you pay it once per document update, not per question.

In [ ]:
import numpy as np

def embed(texts):
    r = with_retry(lambda: client.embeddings.create(model=MODEL_EMBED, input=texts))
    return np.array([d.embedding for d in r.data], dtype=np.float32)

vectors = []
B = 64
for i in range(0, len(chunks), B):
    vectors.append(embed([c["text"] for c in chunks[i:i+B]]))
INDEX = np.vstack(vectors)
INDEX = INDEX / np.linalg.norm(INDEX, axis=1, keepdims=True)   # normalise once
print(f"Index built: {INDEX.shape[0]} chunks × {INDEX.shape[1]} dimensions")

## 3.4 — Retrieve
A question comes in → embed it → find the nearest chunks. That's the whole trick. No database needed at this scale: a matrix multiply does it.

In [ ]:
def retrieve(question, k=4):
    q = embed([question])[0]
    q = q / np.linalg.norm(q)
    sims = INDEX @ q
    top = np.argsort(-sims)[:k]
    return [{"score": float(sims[i]), **chunks[i]} for i in top]

for hit in retrieve("drive keeps tripping F-07 overtemperature, filters are clean"):
    print(f'{hit["score"]:.3f}  {hit["chunk_id"]}')

Look at what came back: the fault-code reference and the service bulletin — exactly the documents a good service engineer would pull. Now wire it into an answer.

In [ ]:
ANSWER_SYSTEM = """You are the Axion Controls service-knowledge assistant.
Answer ONLY from the provided context. Cite sources as [doc-name].
If the context does not contain the answer, say exactly: "Not in the knowledge base — escalate to product support."
Structure: Answer / Why / Source."""

def rag_answer(question, k=4, model=None):
    hits = retrieve(question, k)
    context = "\n\n".join(f'[{h["doc"]}]\n{h["text"]}' for h in hits)
    r = ask(model or MODEL_MINI, [
        {"role": "system", "content": ANSWER_SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    r["sources"] = [h["chunk_id"] for h in hits]
    return r

r = rag_answer("A VectorDrive 450 trips F-07 every hour, filters clean, fan OK, "
               "firmware 2.3, switching at 10 kHz. New fan kit?")
print(r["text"])
print("\nRetrieved:", r["sources"])
print(f'{r["latency_s"]}s · {r["input_tokens"]} in / {r["output_tokens"]} out · ${cost_usd(r)}')

**Compare this with Lab 1, section 1.2** — the same question with no retrieval. The model now says *don't buy the fan kit, upgrade the firmware, here's the bulletin*. Same model. The difference is entirely the context you fed it. That is the lesson of RAG in one sentence: **the model is the engine; your knowledge pipeline is the fuel quality.**

## 3.5 — Break it on purpose
Your teams will hit these three failures in the first month. Meet them here first.

In [ ]:
# BREAK 1 — chunks far too big (whole-document soup)
big = [c for n, t in docs.items() for c in chunk_document(n, t, target_chars=8000)]
vecs = []
for i in range(0, len(big), 64):
    vecs.append(embed([c["text"] for c in big[i:i+64]]))
BIG_INDEX = np.vstack(vecs); BIG_INDEX /= np.linalg.norm(BIG_INDEX, axis=1, keepdims=True)

q = "What does alarm A-12 on the VectorDrive mean?"
qv = embed([q])[0]; qv /= np.linalg.norm(qv)
hits = np.argsort(-(BIG_INDEX @ qv))[:2]
ctx = "\n\n".join(big[i]["text"] for i in hits)
r_big = ask(MODEL_MINI, [{"role": "system", "content": ANSWER_SYSTEM},
                         {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"}])
r_ok  = rag_answer(q)
print(f'8000-char chunks: {r_big["input_tokens"]} input tokens, ${cost_usd(r_big)}')
print(f'1200-char chunks: {r_ok["input_tokens"]} input tokens, ${cost_usd(r_ok)}')
print("\nSame answer quality? Read both:\n\n--- big ---\n", r_big["text"][:300],
      "\n\n--- tuned ---\n", r_ok["text"][:300])

In [ ]:
# BREAK 2 — k too small: the answer needs TWO documents
q = "Our panel shows BadMaxConnectionsReached. How many sessions does the controller support, "\
    "and does any firmware version raise the limit?"
print("--- k=1 ---\n", rag_answer(q, k=1)["text"])
print("\n--- k=5 ---\n", rag_answer(q, k=5)["text"])

In [ ]:
# BREAK 3 — the multi-hop question flat RAG cannot do (keep this one in mind for M5)
q = ("Which customer sites have F-07 tickets on drives affected by SB-2025-04, "
     "and what one action closes all of them?")
print(rag_answer(q, k=5)["text"])
print("\n^ The bulletin and release notes are in the corpus, but the TICKETS are rows in a CSV "
      "the index never saw — and the question needs a JOIN (site x firmware x fault). "
      "Retrieval finds text; it does not join tables. The M5 demo shows what does.")

## 3.6 — Save the index for Lab 4

In [ ]:
np.savez_compressed("rag_index.npz", index=INDEX)
Path("rag_chunks.json").write_text(json.dumps(chunks), encoding="utf-8")
print("Saved rag_index.npz and rag_chunks.json — Lab 4 loads these.")

## 3.7 — Result line
Copy the printed line into the **Paste & check** box for Lab 3 on the course page.

In [ ]:
print(f'LAB3|tokens_tuned={r_ok["input_tokens"]}|tokens_big={r_big["input_tokens"]}')

✏️ **YOUR TURN — the RAG architecture reference, in your own words:**
1. The three dials you saw move answer quality and cost: chunk size, k, and the system prompt. Which one surprised you?
2. What's the *indexing* cost vs the *per-question* cost in this design? Which one scales with usage?
3. Complete: "When my team proposes a RAG project, the first three questions I will ask are …"

---
✅ **Done.** You built a working RAG application and broke it three ways. Lab 4 makes it trustworthy.